# Benchmark efisiensi dan evaluasi detection-count

Notebook ini menjalankan evaluasi yang sama dengan `evaluate_detection_and_benchmark.py` secara bertahap agar mudah direview. Semua model memakai test split, batch size, device, feature count, dan seed yang sama.

Evaluasi utama menggunakan prefix temporal berbasis jumlah detection: 6, 10, 15, 20, 30, dan seluruh detection valid. Notebook mencatat Macro-precision, Macro-recall, Macro-F1, balanced accuracy, accuracy, jumlah objek valid, coverage, jumlah parameter, waktu inferensi, peak memory, ukuran checkpoint, serta metadata waktu training.

In [1]:
from pathlib import Path
import json

import pandas as pd
import torch
import alerce_classifier
from alerce_classifier.models import CNNClassifier, GRUClassifier, HybridCNNGRU
from alerce_classifier.training.loop import (
    benchmark_inference,
    checkpoint_size_bytes,
    count_parameters,
    evaluate,
    evaluate_detection_prefixes,
    load_split,
    make_loader,
)

REPO_ROOT = Path(alerce_classifier.__file__).resolve().parents[2]
ARTIFACT_DIR = REPO_ROOT / 'artifacts'
MODEL_DIRS = {
    'hybrid_cnn_gru': ARTIFACT_DIR,
    'cnn': ARTIFACT_DIR / 'models' / 'cnn',
    'gru': ARTIFACT_DIR / 'models' / 'gru',
}
BENCHMARK_DIR = ARTIFACT_DIR / 'evaluation' / 'benchmark'
PREPROCESSING_DIR = REPO_ROOT / 'data' / 'alerce_ztf_lc_sanchezsaez_full' / 'data' / 'full' / 'preprocessing' / 'feature_experiment'
SPLIT_PATH = PREPROCESSING_DIR / 'test_derived_padded.npz'
OUTPUT_PATH = BENCHMARK_DIR / 'derived_detection_benchmark_results.json'
FEATURE_COUNT = 11
BATCH_SIZE = 64
SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_FACTORIES = {
    'hybrid_cnn_gru': HybridCNNGRU,
    'cnn': CNNClassifier,
    'gru': GRUClassifier,
}
DETECTION_COUNTS = (6, 10, 15, 20, 30, None)
print('Repository:', REPO_ROOT)
print('Device:', DEVICE)
print('Batch size:', BATCH_SIZE)
print('Detection counts:', DETECTION_COUNTS)


Repository: C:\Users\FASILKOMTIUSU\Documents\CLARA_ALERCE\sistem\ztf-classifier
Device: cuda
Batch size: 64
Detection counts: (6, 10, 15, 20, 30, None)


## 1. Validasi artifact dan protocol eksperimen

In [2]:
required = [SPLIT_PATH]
required.extend(
    MODEL_DIRS[model_name] / f'derived_{model_name}_seed{SEED}_best.pt'
    for model_name in MODEL_FACTORIES
)
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(f'Artifact benchmark tidak lengkap: {missing}')

test_x, test_mask, test_y = load_split(SPLIT_PATH)
print('Test tensor:', tuple(test_x.shape))
print('Test object count:', len(test_y))
print('Class counts:', torch.bincount(test_y, minlength=3).tolist())
print('Real sequence length:', {
    'min': int(test_mask.sum(dim=1).min()),
    'mean': float(test_mask.sum(dim=1).float().mean()),
    'max': int(test_mask.sum(dim=1).max()),
})


Test tensor: (18525, 1088, 11)
Test object count: 18525
Class counts: [13060, 5207, 258]
Real sequence length: {'min': 6, 'mean': 410.9811706542969, 'max': 1088}


## 2. Benchmark model dan evaluasi test

Waktu inferensi diukur setelah satu batch warm-up. Pada CUDA, sinkronisasi dilakukan sebelum dan sesudah pengukuran. Pada CPU, peak memory berasal dari `tracemalloc`. Nilai ini perlu dilaporkan sebagai peak memory Python yang terukur, bukan total RAM sistem.

In [3]:
full_loader = make_loader(test_x, test_mask, test_y, batch_size=BATCH_SIZE)
results = {
    'protocol': {
        'seed': SEED,
        'device': str(DEVICE),
        'batch_size': BATCH_SIZE,
        'feature_count': FEATURE_COUNT,
        'detection_counts': ['6', '10', '15', '20', '30', 'all'],
        'test_object_count': int(len(test_y)),
    },
    'models': {},
}
models = {}

for model_name, factory in MODEL_FACTORIES.items():
    checkpoint_path = MODEL_DIRS[model_name] / f'derived_{model_name}_seed{SEED}_best.pt'
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model = factory(feature_count=FEATURE_COUNT).to(DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    models[model_name] = model

    test_metrics = evaluate(model, full_loader, DEVICE)
    inference_metrics = benchmark_inference(model, full_loader, DEVICE)
    early_metrics = evaluate_detection_prefixes(
        model,
        test_x,
        test_mask,
        test_y,
        DETECTION_COUNTS,
        DEVICE,
        batch_size=BATCH_SIZE,
    )
    results['models'][model_name] = {
        'test': test_metrics,
        'early_classification': early_metrics,
        'benchmark': {
            'trainable_parameters': count_parameters(model),
            'checkpoint_size_bytes': checkpoint_size_bytes(checkpoint_path),
            **inference_metrics,
        },
        'selection': {
            'best_epoch': checkpoint.get('best_epoch'),
            'best_validation_macro_f1': checkpoint.get('best_validation_macro_f1'),
            'training_seconds': checkpoint.get('training_seconds'),
        },
    }
    print(model_name, 'completed')


hybrid_cnn_gru completed
cnn completed
gru completed


## 3. Tabel met metrik test dan benchmark efisiensi

In [4]:
test_rows = []
benchmark_rows = []
for model_name, payload in results['models'].items():
    test = payload['test']
    test_rows.append({
        'model': model_name,
        'macro_precision': test['macro_precision'],
        'macro_recall': test['macro_recall'],
        'macro_f1': test['macro_f1'],
        'balanced_accuracy': test['balanced_accuracy'],
        'accuracy': test['accuracy'],
    })
    benchmark = payload['benchmark']
    selection = payload['selection']
    benchmark_rows.append({
        'model': model_name,
        'trainable_parameters': benchmark['trainable_parameters'],
        'training_seconds': selection['training_seconds'],
        'inference_seconds': benchmark['inference_seconds'],
        'inference_seconds_per_object': benchmark['inference_seconds_per_object'],
        'peak_memory_bytes': benchmark['peak_memory_bytes'],
        'checkpoint_size_bytes': benchmark['checkpoint_size_bytes'],
    })

test_table = pd.DataFrame(test_rows).set_index('model')
benchmark_table = pd.DataFrame(benchmark_rows).set_index('model')
display(test_table.style.format('{:.4f}'))
display(benchmark_table)


,macro_precision,macro_recall,macro_f1,balanced_accuracy,accuracy
model,,,,,
hybrid_cnn_gru,0.9638,0.9639,0.9639,0.9639,0.9717
cnn,0.9657,0.9587,0.9621,0.9587,0.9698
gru,0.9763,0.9714,0.9738,0.9714,0.9808


,trainable_parameters,training_seconds,inference_seconds,inference_seconds_per_object,peak_memory_bytes,checkpoint_size_bytes
model,,,,,,
hybrid_cnn_gru,25955,1412.053819,14.104117,0.000761,61990400.0,108469
cnn,7043,102.130655,0.364500,0.000020,48748032.0,31509
gru,14979,NaN,13.770509,0.000743,48989184.0,63189


## 4. Evaluasi prefix temporal berbasis jumlah detection

In [5]:
early_rows = []
for model_name, payload in results['models'].items():
    for detection_count, metrics in payload['early_classification'].items():
        early_rows.append({
            'model': model_name,
            'detection_count': detection_count,
            'valid_object_count': metrics['valid_object_count'],
            'coverage': metrics['coverage'],
            'macro_precision': metrics['macro_precision'],
            'macro_recall': metrics['macro_recall'],
            'macro_f1': metrics['macro_f1'],
            'balanced_accuracy': metrics['balanced_accuracy'],
            'accuracy': metrics['accuracy'],
        })
early_table = pd.DataFrame(early_rows)
display(early_table.style.format({
    'coverage': '{:.2%}',
    'macro_precision': '{:.4f}',
    'macro_recall': '{:.4f}',
    'macro_f1': '{:.4f}',
    'balanced_accuracy': '{:.4f}',
    'accuracy': '{:.4f}',
}))


,model,detection_count,valid_object_count,coverage,macro_precision,macro_recall,macro_f1,balanced_accuracy,accuracy
0,hybrid_cnn_gru,6,18525,100.00%,0.5282,0.7604,0.4912,0.7604,0.6180
1,hybrid_cnn_gru,10,18469,99.70%,0.5609,0.8192,0.5513,0.8192,0.7039
2,hybrid_cnn_gru,15,18294,98.75%,0.5788,0.8473,0.5808,0.8473,0.7585
3,hybrid_cnn_gru,20,18107,97.74%,0.5912,0.8676,0.6002,0.8676,0.8000
4,hybrid_cnn_gru,30,17776,95.96%,0.6061,0.8923,0.6226,0.8923,0.8482
5,hybrid_cnn_gru,all,18525,100.00%,0.9638,0.9639,0.9639,0.9639,0.9717
6,cnn,6,18525,100.00%,0.5243,0.7530,0.4951,0.7530,0.6273
7,cnn,10,18469,99.70%,0.5505,0.8144,0.5489,0.8144,0.6999
8,cnn,15,18294,98.75%,0.5681,0.8456,0.5810,0.8456,0.7542
9,cnn,20,18107,97.74%,0.5800,0.8704,0.5991,0.8704,0.7911


## 5. Simpan hasil

JSON ini menjadi artifact utama untuk tabel Bab 4 dan halaman hasil aplikasi.

In [6]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.write_text(json.dumps(results, indent=2, default=float), encoding='utf-8')
print('Hasil ditulis ke:', OUTPUT_PATH)


Hasil ditulis ke: C:\Users\FASILKOMTIUSU\Documents\CLARA_ALERCE\sistem\ztf-classifier\artifacts\derived_detection_benchmark_results.json
